# W2-D2: RCA

Notebook nay doc cluster output tu W2-D1, chay graph RCA + retrieval, va ghi `results/rca_output.json`.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import networkx as nx

base_dir = Path.cwd()
dataset_dir = base_dir / 'dataset'
results_dir = base_dir / 'results'
results_dir.mkdir(exist_ok=True)

cluster_path = base_dir.parent / 'd1' / 'results' / 'cluster_summary.json'
alerts_path = dataset_dir / 'alerts_sample.jsonl'
graph_path = dataset_dir / 'services.json'
history_path = dataset_dir / 'incidents_history.json'

VALID_CLASSES = {
    'connection_pool_exhaustion', 'slow_query', 'memory_leak', 'rebalance_storm',
    'deadlock', 'network_partition', 'bad_deploy', 'config_push', 'tls_expiry',
    'ddos', 'lock_contention', 'eviction', 'infinite_retry', 'model_drift',
    'rate_limit_misconfig', 'thread_starvation', 'cache_stampede', 'n_plus_1',
    'downstream_provider', 'batch_overlap', 'feature_flag', 'cache_cold_start',
    'replication_lag', 'vacuum_storm', 'other'
}

CLASS_MAP = {label: label for label in VALID_CLASSES if label != 'other'}

def parse_ts(value: str) -> datetime:
    if value.endswith('Z'):
        value = value[:-1] + '+00:00'
    return datetime.fromisoformat(value).astimezone(timezone.utc)

def load_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def load_jsonl(path: Path):
    items = []
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            items.append(json.loads(line))
    return items

def build_graph(payload: dict):
    graph = nx.DiGraph()
    node_types = {}
    for service in payload.get('services', []):
        graph.add_node(service['name'])
        node_types[service['name']] = 'service'
    for store in payload.get('stores', []):
        graph.add_node(store['name'])
        node_types[store['name']] = store.get('type', 'store')
    for edge in payload.get('edges', []):
        source = edge.get('source', edge.get('from'))
        target = edge.get('target', edge.get('to'))
        if source and target:
            graph.add_edge(source, target, edge_type=edge.get('type', 'unknown'))
    return graph, node_types

def normalize_scores(raw_scores: dict[str, float]) -> dict[str, float]:
    if not raw_scores:
        return {}
    max_score = max(raw_scores.values()) or 1.0
    return {node: score / max_score for node, score in raw_scores.items()}

def pagerank_scores(subgraph: nx.DiGraph) -> dict[str, float]:
    if subgraph.number_of_nodes() == 1:
        node = next(iter(subgraph.nodes()))
        return {node: 1.0}
    return normalize_scores(nx.pagerank(subgraph, alpha=0.85))

def timestamp_scores(cluster: dict, alert_lookup: dict[str, dict]) -> dict[str, float]:
    first_seen = {}
    for alert_id in cluster.get('alert_ids', []):
        alert = alert_lookup[alert_id]
        service = alert['service']
        current_ts = parse_ts(alert['ts'])
        if service not in first_seen or current_ts < first_seen[service]:
            first_seen[service] = current_ts
    ordered = sorted(first_seen.items(), key=lambda item: item[1])
    if not ordered:
        return {service: 1.0 for service in cluster['services']}
    if len(ordered) == 1 or ordered[0][1] == ordered[-1][1]:
        return {service: 1.0 for service in first_seen}
    min_ts = ordered[0][1]
    max_ts = ordered[-1][1]
    span = (max_ts - min_ts).total_seconds() or 1.0
    scores = {}
    for service, first_ts in first_seen.items():
        lag = (first_ts - min_ts).total_seconds()
        scores[service] = max(0.0, 1 - (lag / span))
    return scores

def apply_store_refinement(scores: dict[str, float], cluster: dict, alert_lookup: dict[str, dict], node_types: dict[str, str]) -> dict[str, float]:
    if not scores:
        return scores
    top_service = max(scores, key=scores.get)
    if node_types.get(top_service, 'service') == 'service':
        return scores
    store_first_seen = None
    app_first_seen = None
    for alert_id in cluster.get('alert_ids', []):
        alert = alert_lookup[alert_id]
        ts = parse_ts(alert['ts'])
        if alert['service'] == top_service:
            store_first_seen = ts if store_first_seen is None or ts < store_first_seen else store_first_seen
        elif node_types.get(alert['service'], 'service') == 'service':
            app_first_seen = ts if app_first_seen is None or ts < app_first_seen else app_first_seen
    if store_first_seen and app_first_seen and store_first_seen > app_first_seen:
        scores[top_service] *= 0.6
    return scores

def graph_rank_cluster(cluster: dict, graph: nx.DiGraph, node_types: dict[str, str], alert_lookup: dict[str, dict]):
    services = cluster['services']
    subgraph = graph.subgraph(services).copy()
    if subgraph.number_of_nodes() == 0:
        subgraph.add_nodes_from(services)
    pr_scores = pagerank_scores(subgraph)
    ts_scores = timestamp_scores(cluster, alert_lookup)
    final_scores = {}
    for service in services:
        final_scores[service] = 0.6 * pr_scores.get(service, 0.0) + 0.4 * ts_scores.get(service, 0.0)
    final_scores = apply_store_refinement(final_scores, cluster, alert_lookup, node_types)
    ranked = sorted(final_scores.items(), key=lambda item: (-item[1], item[0]))
    return [[service, round(score, 2)] for service, score in ranked[:3]]

def normalize_severity(value: str) -> str:
    mapping = {'crit': 'critical', 'critical': 'critical', 'warn': 'high', 'warning': 'high', 'info': 'low'}
    return mapping.get(value.lower(), value.lower())

def tokenize_cluster(cluster: dict) -> set[str]:
    tokens = set()
    for service in cluster['services']:
        tokens.update(service.replace('-', ' ').replace('_', ' ').split())
    for fingerprint in cluster.get('fingerprints', []):
        tokens.update(fingerprint.lower().replace('|', ' ').replace('_', ' ').replace('-', ' ').split())
    return {token for token in tokens if token}

def tokenize_metric_hints(cluster: dict) -> set[str]:
    hints = set()
    allowed = {'pool', 'connection', 'db', 'query', 'cache', 'queue', 'lock', 'redis', 'kafka', 'timeout'}
    for fingerprint in cluster.get('fingerprints', []):
        parts = fingerprint.lower().split('|')
        if len(parts) >= 2:
            metric_tokens = parts[1].replace('_', ' ').replace('-', ' ').split()
            for token in metric_tokens:
                if token in allowed:
                    hints.add(token)
    return hints

def tokenize_incident(incident: dict) -> set[str]:
    text = ' '.join(incident.get('services_involved', [])) + ' ' + incident.get('summary', '') + ' ' + incident.get('remediation', '')
    tokens = text.lower().replace('-', ' ').replace('_', ' ').replace('/', ' ').split()
    return {token.strip('.,:;()') for token in tokens if token}

def keyword_similarity(cluster_tokens: set[str], incident_tokens: set[str]) -> float:
    if not cluster_tokens or not incident_tokens:
        return 0.0
    intersection = len(cluster_tokens & incident_tokens)
    union = len(cluster_tokens | incident_tokens)
    return intersection / union if union else 0.0

def retrieve_similar(cluster: dict, root_candidate: str, incidents: list[dict]):
    cluster_tokens = tokenize_cluster(cluster)
    metric_hints = tokenize_metric_hints(cluster)
    cluster_services = set(cluster['services'])
    cluster_severity = normalize_severity(cluster['max_severity'])
    ranked = []
    for incident in incidents:
        incident_tokens = tokenize_incident(incident)
        score = 0.0
        if incident['root_cause_service'] in cluster_services:
            score += 0.4
        overlap = len(cluster_services & set(incident.get('services_involved', [])))
        score += min(0.4, 0.2 * overlap)
        if incident.get('severity', '').lower() == cluster_severity:
            score += 0.2
        if incident['root_cause_service'] == root_candidate:
            score += 0.25
        score += 0.2 * keyword_similarity(cluster_tokens, incident_tokens)
        if metric_hints:
            hint_overlap = len(metric_hints & incident_tokens)
            score += min(0.15, 0.05 * hint_overlap)
        if score >= 0.2:
            ranked.append((incident, round(score, 3)))
    ranked.sort(key=lambda item: (-item[1], item[0]['id']))
    return ranked[:3]

def remediation_to_actions(text: str) -> list[str]:
    parts = re.split(r';|\.(?=\s+[A-Z])', text)
    actions = [part.strip() for part in parts if part.strip()]
    return actions[:3] or ['Investigate manually']

def classify_from_retrieval(similar_incidents: list[tuple[dict, float]]):
    if not similar_incidents:
        return 'other', ['Investigate manually'], []
    top_incident, _ = similar_incidents[0]
    mapped_class = CLASS_MAP.get(top_incident['root_cause_class'], 'other')
    actions = remediation_to_actions(top_incident.get('remediation', ''))
    incident_ids = [incident['id'] for incident, _ in similar_incidents]
    return mapped_class, actions, incident_ids

def build_reasoning(root_cause: str, confidence: float, similar_ids: list[str]) -> str:
    if similar_ids:
        return f"{root_cause} ranked #1 (score={confidence:.2f}) by PageRank+timestamp. Similar to {similar_ids[0]}."
    return f"{root_cause} ranked #1 (score={confidence:.2f}) by PageRank+timestamp. No similar incident found."

def build_fallback(cluster: dict, graph_top3: list[list], reason: str) -> dict:
    root_cause = graph_top3[0][0] if graph_top3 else cluster['services'][0]
    confidence = float(graph_top3[0][1]) if graph_top3 else 0.0
    return {
        'cluster_id': cluster['cluster_id'],
        'graph_top3': graph_top3 or [[root_cause, round(confidence, 2)]],
        'root_cause': root_cause,
        'class': 'other',
        'confidence': round(max(0.0, min(confidence, 1.0)), 2),
        'actions': ['Investigate manually'],
        'reasoning': reason,
        'similar_incidents': [],
        'method': 'graph-only-fallback',
    }

def validate_result(result: dict, cluster: dict):
    if not result.get('graph_top3'):
        return False, 'graph_top3 empty'
    if not isinstance(result.get('similar_incidents', []), list):
        return False, 'similar_incidents not list'
    if not isinstance(result.get('actions', []), list) or not result.get('actions'):
        return False, 'actions invalid'
    if not isinstance(result.get('reasoning', ''), str) or not result.get('reasoning', '').strip():
        return False, 'reasoning empty'
    if result.get('root_cause') not in cluster['services']:
        return False, 'root_cause outside cluster'
    if result.get('class') not in VALID_CLASSES:
        return False, 'class outside enum'
    confidence = result.get('confidence')
    if not isinstance(confidence, (int, float)) or not (0.0 <= confidence <= 1.0):
        return False, 'confidence outside range'
    return True, 'ok'

clusters_payload = load_json(cluster_path)
alerts = load_jsonl(alerts_path)
services_payload = load_json(graph_path)
history_payload = load_json(history_path)

clusters = clusters_payload['clusters']
alert_lookup = {alert['id']: alert for alert in alerts}
graph, node_types = build_graph(services_payload)
incidents = history_payload['incidents']

print('cluster keys:', clusters[0].keys())
print('alert keys:', alerts[0].keys())
print('services payload keys:', services_payload.keys())
print('incident keys:', incidents[0].keys())

cluster keys: dict_keys(['cluster_id', 'alert_count', 'services', 'time_range', 'max_severity', 'fingerprints', 'alert_ids'])
alert keys: dict_keys(['id', 'ts', 'service', 'metric', 'severity', 'value', 'threshold', 'labels'])
services payload keys: dict_keys(['_meta', 'services', 'stores', 'edges', 'topology_notes'])
incident keys: dict_keys(['id', 'ts', 'severity', 'services_involved', 'root_cause_service', 'root_cause_class', 'summary', 'remediation', 'mttd_min', 'mttr_min'])


In [2]:
graph_core = []
for cluster in clusters:
    graph_top3 = graph_rank_cluster(cluster, graph, node_types, alert_lookup)
    graph_core.append({'cluster_id': cluster['cluster_id'], 'graph_top3': graph_top3})

largest_cluster = max(clusters, key=lambda item: item['alert_count'])
largest_graph = next(item for item in graph_core if item['cluster_id'] == largest_cluster['cluster_id'])
print('largest cluster:', largest_cluster['cluster_id'])
print('graph_top3:', largest_graph['graph_top3'])

largest cluster: c-001-000
graph_top3: [['payment-svc', 0.98], ['checkout-svc', 0.81], ['cart-svc', 0.58]]


In [3]:
results = []
for cluster in clusters:
    graph_top3 = graph_rank_cluster(cluster, graph, node_types, alert_lookup)
    root_cause = graph_top3[0][0] if graph_top3 else cluster['services'][0]
    confidence = float(graph_top3[0][1]) if graph_top3 else 0.0
    similar = retrieve_similar(cluster, root_cause, incidents)
    root_class, actions, similar_ids = classify_from_retrieval(similar)
    reasoning = build_reasoning(root_cause, confidence, similar_ids)
    result = {
        'cluster_id': cluster['cluster_id'],
        'graph_top3': graph_top3,
        'root_cause': root_cause,
        'class': root_class,
        'confidence': round(max(0.0, min(confidence, 1.0)), 2),
        'actions': actions,
        'reasoning': reasoning,
        'similar_incidents': similar_ids,
        'method': 'graph+retrieval',
    }
    valid, reason = validate_result(result, cluster)
    if not valid:
        result = build_fallback(cluster, graph_top3, f'Fallback because {reason}.')
    results.append(result)

rca_output = {'clusters_analyzed': len(results), 'results': results}
print(json.dumps(rca_output, indent=2))
print('--- retrieval smoke ---')
print(results[0]['similar_incidents'])
print(results[0]['class'])
print(results[0]['actions'])

{
  "clusters_analyzed": 4,
  "results": [
    {
      "cluster_id": "c-001-000",
      "graph_top3": [
        [
          "payment-svc",
          0.98
        ],
        [
          "checkout-svc",
          0.81
        ],
        [
          "cart-svc",
          0.58
        ]
      ],
      "root_cause": "payment-svc",
      "class": "connection_pool_exhaustion",
      "confidence": 0.98,
      "actions": [
        "Rollback to v3.1",
        "Scale pool 50 -> 100 cushion",
        "Add pool monitor alert > 80%."
      ],
      "reasoning": "payment-svc ranked #1 (score=0.98) by PageRank+timestamp. Similar to INC-2025-11-08.",
      "similar_incidents": [
        "INC-2025-11-08",
        "INC-2025-09-05",
        "INC-2026-05-10"
      ],
      "method": "graph+retrieval"
    },
    {
      "cluster_id": "c-001-001",
      "graph_top3": [
        [
          "notification-svc",
          1.0
        ]
      ],
      "root_cause": "notification-svc",
      "class": "downstream_p

In [4]:
output_path = results_dir / 'rca_output.json'
output_path.write_text(json.dumps(rca_output, indent=2), encoding='utf-8')
print(output_path)
for result in rca_output['results']:
    print(result['cluster_id'], result['root_cause'], result['confidence'], result['similar_incidents'][:2])

C:\Users\Admin\aiops-Thinh\w2\d2\results\rca_output.json
c-001-000 payment-svc 0.98 ['INC-2025-11-08', 'INC-2025-09-05']
c-001-001 notification-svc 1.0 ['INC-2026-02-08', 'INC-2026-05-18']
c-001-002 recommender-svc 1.0 ['INC-2026-03-07', 'INC-2026-06-02']
c-001-003 search-svc 1.0 ['INC-2026-01-29', 'INC-2026-05-25']
